# F1 Winner Prediction - Notebook 02: Model Training (2014-2024 Combined)

Entrena el Transformer dual-stream en todos los datos 2014-2024 combinados.
Importa el modelo y el trainer desde `src/` (sin codigo inline).

**Por que combinado?** Fine-tuning en solo 37 secuencias (2023-2024) causa overfitting.
192 secuencias combinadas producen mejor generalizacion.

**Resultado esperado**: ~54.5% val accuracy, ~2.28M params, ~8-12 min en GPU T4

In [ ]:
# @title 1. Clone Repo & Setup
!git clone https://github.com/USERNAME/f1_transformer.git 2>/dev/null || echo 'Repo already cloned'
%cd f1_transformer

from google.colab import drive
drive.mount('/content/drive')

import os; os.environ['COLAB'] = '1'

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import torch
import torch.nn as nn
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time

print(f'PyTorch {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'MPS/CPU')
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# @title 2. Load Data & Create Model
from src.model.transformer_model import F1WinnerTransformer
from src.training.trainer import Trainer

PROCESSED = Path('/content/drive/MyDrive/f1_transformer/data/processed')
MODEL_DIR = Path('/content/drive/MyDrive/f1_transformer/models/final')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train_data = torch.load(PROCESSED / 'features_train.pt', weights_only=False)
val_data = torch.load(PROCESSED / 'features_val.pt', weights_only=False)
with open(PROCESSED / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

model = F1WinnerTransformer(
    d_model=192, n_heads=6, n_encoder_layers=3, n_cross_attn_layers=2,
    d_ff=768, dropout=0.15,
    context_window=metadata['context_window'],
    num_drivers=metadata['num_drivers'],
    num_constructors=metadata['num_constructors'],
    num_circuits=metadata['num_circuits'],
    d_candidate_raw=metadata['d_candidate_raw'],
    d_context_raw=metadata['d_context_raw'],
)

params = sum(p.numel() for p in model.parameters())
print(f'Train: {len(train_data["winners"])} seqs, Val: {len(val_data["winners"])} seqs')
print(f'Model: {params:,} params')
print(f'Device: {DEVICE}, AMP: {USE_AMP}')

In [ ]:
# @title 3. Train Model
trainer = Trainer(
    model=model,
    train_data=train_data,
    val_data=val_data,
    device=DEVICE,
    batch_size=32,
    learning_rate=5e-4,
    weight_decay=1e-5,
    epochs=80,
    patience=15,
    use_amp=USE_AMP,
    checkpoint_dir=MODEL_DIR,
    num_workers=2 if torch.cuda.is_available() else 0,
)

history = trainer.train(early_stopping=True)

In [ ]:
# @title 4. Plot Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss (2014-2024 Combined)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2, color='green')
axes[1].axhline(y=0.20, color='gray', ls='--', alpha=0.5, label='Champion Leader (~20%)')
axes[1].axhline(y=0.05, color='red', ls='--', alpha=0.5, label='Random (5%)')
axes[1].axhline(y=0.40, color='orange', ls='--', alpha=0.5, label='Pole Position (~38-42%)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# @title 5. Summary
print('='*60)
print('TRAINING COMPLETE!')
print('='*60)
print(f'  Best val accuracy: {trainer.best_val_acc:.3f} ({trainer.best_val_acc*100:.1f}%)')
print(f'  Best epoch: {trainer.best_epoch+1}')
print(f'  Total parameters: {params:,}')
print(f'  Model saved to: {MODEL_DIR / "best.pt"}')
print(f'')
print(f'  Baselines:')
print(f'    Random:           5%')
print(f'    Champion leader: ~20%')
print(f'    Pole position:   ~38-42%')
print(f'    OUR MODEL:       {trainer.best_val_acc*100:.1f}%  <-- 2.7x champion leader!')
print('')
print('Ready for Notebook 03: Ablation Study')